# 🔬 Cross-modal Distillation — Signformer (student) ← Skeleton (teacher)

Finetuna il **Signformer 40%** con distillazione **FD-CMKD** (feature-only) dal teacher skeleton.
Tutti i parametri sono **qui nella cella `PARAMS`** — modificali e rilancia.

**Tutto è in questa cartella `distillation/`** (modelli inclusi). Prerequisiti sul server:
1. il repo Signformer (per `main/`) e i **dati I3D** in `data/PHOENIX2014T/phoenix14t.pami0.{train,dev,test}`
2. le feature teacher in `distillation/skeleton_feats/` (vedi cella "Estrazione" se mancano)

Output in `distillation/sign_distill/` — il 40% (`distillation/student_40.ckpt`) resta intatto.

## 0 — Setup (portati alla root di Signformer)

In [ ]:
import os, sys
# cwd = code/distillation (dove sta il notebook)
DIST_ROOT = os.path.abspath(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd())!='distillation' else os.getcwd())
if os.path.basename(DIST_ROOT) != 'distillation':
    # fallback: risali finche trovi la struttura code/distillation
    p = os.getcwd()
    while p and os.path.basename(p) != 'distillation':
        p = os.path.dirname(p)
    DIST_ROOT = p
os.chdir(DIST_ROOT)
# aggiungi code/signformer al sys.path per 'import main.training'
SIGNFORMER = os.path.abspath(os.path.join(DIST_ROOT, '..', 'signformer'))
sys.path.insert(0, SIGNFORMER)      # per: import main.training as mt
sys.path.insert(0, os.path.dirname(DIST_ROOT))  # per: from distillation.distill import ...
print('cwd:', os.getcwd())
print('signformer path:', SIGNFORMER)
assert os.path.isdir(os.path.join(SIGNFORMER, 'main')), 'code/signformer/main not found'


cwd: /home/ebufi/phoenix/Signformer


## 1 — ⚙️ PARAMETRI  (modifica qui e rilancia)

In [ ]:
PARAMS = {
    # --- percorsi (relativi alla root di Signformer) ---
    'student_ckpt':      '/home/ebufi/phoenix/Signformer/../signformer/sign_sample/best.ckpt',   # Signformer 40% da cui partire
    'teacher_feats_dir': '/home/ebufi/phoenix/Signformer/../../dataset/features/skeleton_feats',    # feature teacher precalcolate
    'model_dir':         '/home/ebufi/phoenix/Signformer/distillation/run_lam2',      # output (non tocca il 40%)

    # --- distillazione (le leve principali) ---
    'enabled':  True,     # False = training normale (controllo, senza distillazione)
    'lambda':   2.0,      # peso globale della distillazione
    'low_w':    1.0,      # basse frequenze (MSE)  -> semantica condivisa
    'high_w':   0.25,      # alte frequenze (logMSE) -> dettagli specifici

    # --- training ---
    'learning_rate':           0.00005,
    'batch_size':              32,
    'validation_freq':         100,
    'early_stopping_patience': 10,
}
PARAMS

{'student_ckpt': '/home/ebufi/phoenix/Signformer/distillation/student_40.ckpt',
 'teacher_feats_dir': '/home/ebufi/phoenix/Signformer/distillation/skeleton_feats',
 'model_dir': '/home/ebufi/phoenix/Signformer/distillation/run_lam2',
 'enabled': True,
 'lambda': 2.0,
 'low_w': 1.0,
 'high_w': 0.25,
 'learning_rate': 5e-05,
 'batch_size': 32,
 'validation_freq': 100,
 'early_stopping_patience': 10}

## 2 — Controlli di sanità (checkpoint + feature teacher)

In [ ]:
import os, pickle
assert os.path.exists(PARAMS['student_ckpt']), f"manca il checkpoint student: {PARAMS['student_ckpt']}"
print('student ckpt OK:', PARAMS['student_ckpt'])

tf = os.path.join(PARAMS['teacher_feats_dir'], 'train.pkl')
if os.path.exists(tf):
    d = pickle.load(open(tf, 'rb')); k = next(iter(d))
    print(f"teacher feats OK: {len(d)} video | dim={d[k].shape[1]} | es. T={d[k].shape[0]}")
else:
    print('⚠️  feature teacher MANCANTI ->', tf)
    print('   Genera con la cella "Estrazione" in fondo, oppure extract_skeleton_feats.py')

student ckpt OK: /home/ebufi/phoenix/Signformer/distillation/student_40.ckpt
teacher feats OK: 7096 video | dim=512 | es. T=86


## 3 — Costruisci la config attiva dai PARAMS

In [ ]:
import yaml
with open('/home/ebufi/phoenix/Signformer/distillation/sign_distill.yaml') as f:
    cfg = yaml.safe_load(f)

tr = cfg['training']
tr['load_model']              = PARAMS['student_ckpt']
tr['model_dir']               = PARAMS['model_dir']
tr['learning_rate']           = PARAMS['learning_rate']
tr['batch_size']              = PARAMS['batch_size']
tr['validation_freq']         = PARAMS['validation_freq']
tr['early_stopping_patience'] = PARAMS['early_stopping_patience']
tr['distillation'] = {
    'enabled':           PARAMS['enabled'],
    'teacher_feats_dir': PARAMS['teacher_feats_dir'],
    'lambda':            PARAMS['lambda'],
    'low_w':             PARAMS['low_w'],
    'high_w':            PARAMS['high_w'],
}

ACTIVE_CFG = '/home/ebufi/phoenix/Signformer/distillation/active_sign_distill.yaml'
with open(ACTIVE_CFG, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)
print('config attiva scritta ->', ACTIVE_CFG)
print(f"  distillazione={PARAMS['enabled']} | lambda={PARAMS['lambda']} low_w={PARAMS['low_w']} high_w={PARAMS['high_w']}")
print(f"  output -> {PARAMS['model_dir']}")

config attiva scritta -> /home/ebufi/phoenix/Signformer/distillation/active_sign_distill.yaml
  distillazione=True | lambda=2.0 low_w=1.0 high_w=0.25
  output -> /home/ebufi/phoenix/Signformer/distillation/run_lam2


## 3.5 — Optuna hyperparameter search (optional, ~30% epochs)

In [ ]:
# If USE_OPTUNA=True, run a short Optuna study on ~30% of the effective training length
# (via reduced early_stopping_patience) to find (lambda, high_w, lr); then update PARAMS
# and rewrite the active YAML. If False, keep the current PARAMS values.
USE_OPTUNA   = True
OPTUNA_TRIALS = 8
OPTUNA_FRAC   = 0.30

if USE_OPTUNA:
    try:
        import optuna
    except ImportError:
        import subprocess, sys as _s
        subprocess.check_call([_s.executable, '-m', 'pip', 'install', '-q', 'optuna'])
        import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    import copy, gc, tempfile, torch
    import main.training as mt
    import distillation.distill_trainer as DT
    from distillation.distill import load_teacher_feats
    # cache teacher features once
    _TF_PATH = os.path.join(PARAMS['teacher_feats_dir'], 'train.pkl')
    _TF = load_teacher_feats(_TF_PATH)
    DT.load_teacher_feats = lambda p: _TF

    with open('sign_distill.yaml') as f:
        _base_cfg = yaml.safe_load(f)
    _train_data, _dev_data, _test_data, _gls_vocab, _txt_vocab = mt.load_data(data_cfg=_base_cfg['data'])
    _sgn_dim = (sum(_base_cfg['data']['feature_size'])
                if isinstance(_base_cfg['data']['feature_size'], list)
                else _base_cfg['data']['feature_size'])
    _short_patience = max(3, int(PARAMS['early_stopping_patience'] * OPTUNA_FRAC))

    def _trial_wer(trial):
        lam    = trial.suggest_float('lambda', 0.1, 3.0, log=True)
        high_w = trial.suggest_float('high_w', 0.0, 1.0)
        lr     = trial.suggest_float('lr', 2e-5, 2e-4, log=True)
        cfg_t = copy.deepcopy(_base_cfg)
        tr = cfg_t['training']
        tr['load_model']              = PARAMS['student_ckpt']
        tr['learning_rate']           = float(lr)
        tr['batch_size']              = PARAMS['batch_size']
        tr['validation_freq']         = PARAMS['validation_freq']
        tr['early_stopping_patience'] = _short_patience
        tr['reset_best_ckpt'] = True; tr['reset_optimizer'] = True; tr['reset_scheduler'] = True
        tr['overwrite'] = True
        tr['model_dir'] = tempfile.mkdtemp(prefix='optuna_trial_', dir='.')
        tr['distillation'] = {'enabled': True,
                              'teacher_feats_dir': PARAMS['teacher_feats_dir'],
                              'lambda': float(lam), 'low_w': PARAMS['low_w'],
                              'high_w': float(high_w)}
        do_rec = tr.get('recognition_loss_weight', 1.0) > 0.0
        do_trs = tr.get('translation_loss_weight', 1.0) > 0.0
        mm = cfg_t['data'].get('multimodal', 1.0) > 0.0
        _model = mt.build_model(cfg=cfg_t['model'], multimodal=mm,
                                gls_vocab=_gls_vocab, txt_vocab=_txt_vocab, sgn_dim=_sgn_dim,
                                do_recognition=do_rec, do_translation=do_trs)
        trainer = DT.DistillTrainManager(model=_model, config=cfg_t)
        trainer.train_and_validate(train_data=_train_data, valid_data=_dev_data)
        wer = float(trainer.best_ckpt_score)
        del trainer, _model
        gc.collect(); torch.cuda.empty_cache()
        return wer

    study = optuna.create_study(direction='minimize',
                                sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(_trial_wer, n_trials=OPTUNA_TRIALS, show_progress_bar=False)
    print('[Optuna] best dev WER:', round(study.best_value, 2))
    print('[Optuna] best params :', study.best_params)
    PARAMS['lambda']        = study.best_params['lambda']
    PARAMS['high_w']        = study.best_params['high_w']
    PARAMS['learning_rate'] = study.best_params['lr']
    print('PARAMS updated with Optuna best.')
else:
    print('Optuna search disabled — using current PARAMS values.')


## 4 — ▶️ Finetuning con distillazione

In [ ]:
import sys
import os

# Aggiungi la cartella 'Signformer' al path
# Assumendo che il notebook sia in 'Signformer/distillation'
# Sostituisci la riga precedente con questa (percorso fisso)
project_root = '/home/ebufi/phoenix/Signformer'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Ora puoi importare senza errori dal file distill_trainer.py
from distillation.distill_trainer import train_distill

2026-07-04 21:11:50.941007: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-07-04 21:11:51.435948: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
2026-07-04 21:11:52.107713: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:995] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-07-04 21:11:52.130565: W tensorflow/core/common_runtime/gpu/gpu_device.cc:1960] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GP

In [ ]:
from distillation.distill_trainer import train_distill
train_distill(ACTIVE_CFG)


## 5 — Grafici (loss + WER)
Il WER stampato è quello dello **student** (Signformer): obiettivo scendere **sotto 40%**.

In [ ]:
import main.training as mt
png = mt.plot_training_curves(PARAMS['model_dir'])
print('grafici:', png)
from IPython.display import Image
Image(filename=png) if png and os.path.exists(png) else print('nessun grafico ancora')

## (opzionale) Estrazione feature teacher — solo se `skeleton_feats/` manca
Richiede i keypoint MSKA e `tssi75_cslr_best.pt` (gia' in questa cartella).
Modifica `DATA_DIR` dentro lo script se i keypoint sono altrove.

In [ ]:
!python extract_skeleton_feats.py


teacher caricato | num_classes=1022 | feat_dim=512 | dev=cuda
  [train] 500/7096
  [train] 1000/7096
  [train] 1500/7096
  [train] 2000/7096
  [train] 2500/7096
  [train] 3000/7096
  [train] 3500/7096
  [train] 4000/7096
  [train] 4500/7096
  [train] 5000/7096
  [train] 5500/7096
  [train] 6000/7096
  [train] 6500/7096
  [train] 7000/7096
[train] 7096 feature salvate -> skeleton_feats/train.pkl
  [dev] 500/519
[dev] 519 feature salvate -> skeleton_feats/dev.pkl
  [test] 500/642
[test] 642 feature salvate -> skeleton_feats/test.pkl
FATTO. feature teacher pronte in skeleton_feats
